# Time Series Momentum vs Optimal-Trade Random Forest

This notebook compares two equities strategies on the same 2020-to-now test window:

- A paper-style Time Series Momentum implementation inspired by Moskowitz, Ooi, and Pedersen (2012): sign of the prior 12-month return, rebalanced monthly, with inverse-volatility scaling.
- A pre-2020 Random Forest classifier trained on pooled oracle optimal trades for `YE = [1, 2, 4, 8]` with `min_profit_pct = 5%`.

Universe note:

- The original paper studies futures and excess returns. This notebook now uses a screened US equities universe with market cap >= $10B from the local FMP-backed database, so it is a large-cap equities test rather than a futures replication.


In [1]:
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import django
from django.apps import apps

os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "settings")
if not apps.ready:
    django.setup()

from backtest.backtest import ExecutionConfig, backtest_panel
from backtest.strategies.benchmark import BuyAndHoldEqualWeightStrategy
from ml.models import ModelArtifact
from pipeline.optimal_trade_random_forest import (
    build_optimal_trade_feature_frame,
    run_optimal_trade_random_forest_training,
    score_optimal_trade_random_forest_models,
)
from pipeline.universe_selection import DEFAULT_US_EXCHANGES, resolve_symbol_universe

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [2]:
EQUITY_UNIVERSE_CONFIG = {
    "country": "US",
    "exchanges": list(DEFAULT_US_EXCHANGES),
    "min_market_cap": 10_000_000_000.0,
    "exclude_pooled_vehicles": True,
}

SCREENED_EQUITY_SYMBOLS = resolve_symbol_universe(
    min_market_cap=float(EQUITY_UNIVERSE_CONFIG["min_market_cap"]),
    country=EQUITY_UNIVERSE_CONFIG["country"],
    exchanges=EQUITY_UNIVERSE_CONFIG["exchanges"],
    exclude_pooled_vehicles=bool(EQUITY_UNIVERSE_CONFIG["exclude_pooled_vehicles"]),
)
EQUITY_UNIVERSE_DIAGNOSTICS = pd.DataFrame(
    [
        {
            "country": EQUITY_UNIVERSE_CONFIG["country"],
            "exchanges": ", ".join(EQUITY_UNIVERSE_CONFIG["exchanges"]),
            "min_market_cap": float(EQUITY_UNIVERSE_CONFIG["min_market_cap"]),
            "exclude_pooled_vehicles": bool(EQUITY_UNIVERSE_CONFIG["exclude_pooled_vehicles"]),
            "resolved_symbol_count": int(len(SCREENED_EQUITY_SYMBOLS)),
            "sample_symbols": ", ".join(SCREENED_EQUITY_SYMBOLS[:25]),
        }
    ]
)

NOTEBOOK_CONFIG = {
    "symbols": SCREENED_EQUITY_SYMBOLS,
    "train_start_date": "1900-01-01",
    "train_end_date": "2019-12-31",
    "backtest_start_date": "2020-01-01",
    "backtest_end_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
    "rebalance_freq": "M",
    "fee_bps": 2.0,
    "slippage_bps": 8.0,
    "gross_exposure": 1.0,
    "fetch_missing_from_fmp": True,
    "rf_model_name_prefix": "optimal_trade_rf_us_equity_10b_pre2020_min1",
    "rf_ye_list": [1, 2, 4, 8],
    "rf_min_profit_pct": 1.0,
    "rf_min_feature_coverage_pct": 10.0,
    "train_rf_if_missing": True,
    "tsmom_signal_col_candidates": ["px__ret_252d", "px__ret_252_d"],
    "tsmom_vol_col_candidates": ["px__vol_63", "px__vol_20", "vol_5"],
    "output_basename": "notebook_tsmom_vs_rf_us_equity_10b",
}

display(Markdown("### Equity Universe Screen"))
display(EQUITY_UNIVERSE_DIAGNOSTICS)
display(Markdown("### Notebook Config"))
display(pd.DataFrame([NOTEBOOK_CONFIG]).T.rename(columns={0: "value"}))


### Equity Universe Screen

,country,exchanges,min_market_cap,exclude_pooled_vehicles,resolved_symbol_count,sample_symbols
0,US,"NASDAQ, NYSE, AMEX",1.000000e+10,True,762,"GOOG, AVGO, BRK-B, BRK-A, WMT, LLY, JPM, XOM, ..."


### Notebook Config

,value
symbols,"[GOOG, AVGO, BRK-B, BRK-A, WMT, LLY, JPM, XOM,..."
train_start_date,1900-01-01
train_end_date,2019-12-31
backtest_start_date,2020-01-01
backtest_end_date,2026-03-19
rebalance_freq,M
fee_bps,2.0
slippage_bps,8.0
gross_exposure,1.0
fetch_missing_from_fmp,True


In [3]:
def first_available_column(columns, candidates):
    column_set = set(columns)
    for candidate in candidates:
        if candidate in column_set:
            return candidate
    return None


def _normalize_symbol_list(values):
    return sorted({str(value).strip().upper() for value in list(values or []) if str(value).strip()})


def _load_existing_rf_summary(config):
    output_name = str(config.get("rf_model_name_prefix") or config.get("output_basename") or "").strip()
    summary_path = Path("data/pipeline_artifacts") / f"{output_name}__summary.json"
    if not summary_path.exists():
        return {}
    try:
        return json.loads(summary_path.read_text())
    except Exception:
        return {}


def _rf_config_mismatch_reasons(existing_summary, config):
    if not isinstance(existing_summary, dict) or not existing_summary:
        return ["missing_summary"]
    reasons = []
    existing_symbols = _normalize_symbol_list(existing_summary.get("symbols_requested"))
    config_symbols = _normalize_symbol_list(config.get("symbols"))
    if existing_symbols != config_symbols:
        reasons.append("symbols")
    if str(existing_summary.get("feature_start_date") or "") != str(config.get("train_start_date") or ""):
        reasons.append("train_start_date")
    if str(existing_summary.get("train_end_date") or "") != str(config.get("train_end_date") or ""):
        reasons.append("train_end_date")
    existing_ye = [int(value) for value in list(existing_summary.get("ye_list") or [])]
    config_ye = [int(value) for value in list(config.get("rf_ye_list") or [])]
    if existing_ye != config_ye:
        reasons.append("ye_list")
    try:
        existing_profit = float(existing_summary.get("min_profit_pct_points"))
    except Exception:
        existing_profit = None
    try:
        config_profit = float(config.get("rf_min_profit_pct"))
    except Exception:
        config_profit = None
    if existing_profit != config_profit:
        reasons.append("min_profit_pct")
    try:
        existing_cov = float(existing_summary.get("min_feature_coverage_pct"))
    except Exception:
        existing_cov = None
    try:
        config_cov = float(config.get("rf_min_feature_coverage_pct", 10.0))
    except Exception:
        config_cov = None
    if existing_cov != config_cov:
        reasons.append("min_feature_coverage_pct")
    return reasons


def ensure_rf_models(config):
    classifier_name = f"{config['rf_model_name_prefix']}_classifier"
    classifier_exists = ModelArtifact.objects.filter(name=classifier_name).exists()
    existing_summary = _load_existing_rf_summary(config)
    mismatch_reasons = _rf_config_mismatch_reasons(existing_summary, config) if classifier_exists else ["missing_model"]
    needs_retrain = bool(mismatch_reasons)
    availability_df = pd.DataFrame(
        [
            {
                "ye_list": [int(k) for k in config["rf_ye_list"]],
                "classifier_name": classifier_name,
                "classifier_exists": bool(classifier_exists),
                "needs_retrain": bool(needs_retrain),
                "mismatch_reasons": ", ".join(mismatch_reasons),
            }
        ]
    )
    if not needs_retrain:
        return {"status": "already_available", "missing_names": [], "mismatch_reasons": []}, availability_df
    if not bool(config.get("train_rf_if_missing", False)):
        raise RuntimeError(f"RF artifacts are missing or stale for {classifier_name}: {', '.join(mismatch_reasons)}")
    training_summary = run_optimal_trade_random_forest_training(
        symbols=config["symbols"],
        feature_start_date=config["train_start_date"],
        train_end_date=config["train_end_date"],
        ye_list=config["rf_ye_list"],
        min_profit_pct=float(config["rf_min_profit_pct"]),
        min_feature_coverage_pct=float(config.get("rf_min_feature_coverage_pct", 10.0)),
        output_basename=config["rf_model_name_prefix"],
        model_name_prefix=config["rf_model_name_prefix"],
        download_missing_prices=bool(config.get("fetch_missing_from_fmp", True)),
    )
    return training_summary, availability_df


def load_rf_training_artifacts(config, training_summary):
    output_name = str(config.get("rf_model_name_prefix") or config.get("output_basename") or "").strip()
    artifact_dir = Path("data/pipeline_artifacts")
    summary_path = artifact_dir / f"{output_name}__summary.json"
    coverage_path = artifact_dir / f"{output_name}__feature_coverage.csv"
    label_path = artifact_dir / f"{output_name}__labels.csv"
    summary_payload = training_summary if isinstance(training_summary, dict) and training_summary.get("model_training_summary") else {}
    if not summary_payload and summary_path.exists():
        summary_payload = json.loads(summary_path.read_text())
    feature_coverage = pd.read_csv(coverage_path) if coverage_path.exists() else pd.DataFrame()
    if not label_path.exists() and isinstance(summary_payload, dict):
        uri = str(summary_payload.get("label_frame_uri") or "").strip()
        if uri:
            candidate = Path(uri)
            if candidate.exists():
                label_path = candidate
    label_frame = pd.read_csv(label_path) if label_path.exists() else pd.DataFrame()
    return summary_payload, feature_coverage, label_frame


def build_rf_training_objective_table(config, summary_payload):
    model_summary = (summary_payload or {}).get("model_training_summary", {})
    return pd.DataFrame(
        [
            {
                "train_window": f"{config['train_start_date']} to {config['train_end_date']}",
                "backtest_window": f"{config['backtest_start_date']} to {config['backtest_end_date']}",
                "symbols_requested": int(len(config["symbols"])),
                "ye_list": [int(k) for k in config["rf_ye_list"]],
                "min_profit_pct": float(config["rf_min_profit_pct"]),
                "min_feature_coverage_pct": float(config.get("rf_min_feature_coverage_pct", np.nan)),
                "training_rows": int(model_summary.get("training_rows", 0) or 0),
                "feature_count": int(model_summary.get("feature_count", 0) or 0),
                "classifier_target": "oracle trade side (long=1, short=0)",
                "signal_definition": str(model_summary.get("signal_definition", "2 * P(long) - 1")),
                "label_source": "pooled pre-2020 optimal trades across YE horizons",
                "execution_mapping": "long trades use buy->sell; short trades use short->cover",
            }
        ]
    )


def build_rf_label_diagnostics_table(label_frame, summary_payload):
    trade_stats = dict(((summary_payload or {}).get("label_statistics") or {}).get("trade_stats") or {})
    labels = label_frame.copy() if label_frame is not None else pd.DataFrame()
    if labels.empty:
        return pd.DataFrame(
            [
                {
                    "label_rows_total": int((summary_payload or {}).get("label_row_count", 0) or 0),
                    "label_rows_pooled_ye": 0,
                    "unique_date_symbol_rows": 0,
                    "duplicate_rows_beyond_first": 0,
                    "max_rows_per_date_symbol": 0,
                    "long_labels": int(trade_stats.get("long_trades", 0) or 0),
                    "short_labels": int(trade_stats.get("short_trades", 0) or 0),
                    "symbols_with_labels": int(trade_stats.get("symbols_count", 0) or 0),
                }
            ]
        )
    labels["date"] = pd.to_datetime(labels.get("date"), errors="coerce")
    labels["symbol"] = labels.get("symbol").astype(str).str.strip().str.upper()
    labels["freq"] = labels.get("freq").astype(str).str.upper()
    labels["k"] = pd.to_numeric(labels.get("k"), errors="coerce")
    labels = labels.dropna(subset=["date", "symbol"]).copy()
    pooled = labels[
        (labels["freq"] == "YE")
        & (labels["k"].isin([1, 2, 4, 8]))
    ].copy()
    dup_counts = pooled.groupby(["date", "symbol"]).size() if not pooled.empty else pd.Series(dtype=int)
    return pd.DataFrame(
        [
            {
                "label_rows_total": int(len(labels)),
                "label_rows_pooled_ye": int(len(pooled)),
                "unique_date_symbol_rows": int(len(dup_counts)),
                "duplicate_rows_beyond_first": int((dup_counts - 1).clip(lower=0).sum()) if len(dup_counts) else 0,
                "max_rows_per_date_symbol": int(dup_counts.max()) if len(dup_counts) else 0,
                "long_labels": int((pooled.get("label") == 1).sum()) if "label" in pooled.columns else int(trade_stats.get("long_trades", 0) or 0),
                "short_labels": int((pooled.get("label") == 0).sum()) if "label" in pooled.columns else int(trade_stats.get("short_trades", 0) or 0),
                "symbols_with_labels": int(pooled["symbol"].nunique()) if not pooled.empty else int(trade_stats.get("symbols_count", 0) or 0),
            }
        ]
    )


def build_rf_label_breakdown_tables(label_frame, config):
    labels = label_frame.copy() if label_frame is not None else pd.DataFrame()
    if labels.empty:
        empty = pd.DataFrame()
        return empty, empty, empty
    labels["date"] = pd.to_datetime(labels.get("date"), errors="coerce")
    labels["symbol"] = labels.get("symbol").astype(str).str.strip().str.upper()
    labels["freq"] = labels.get("freq").astype(str).str.upper()
    labels["k"] = pd.to_numeric(labels.get("k"), errors="coerce")
    labels["label"] = pd.to_numeric(labels.get("label"), errors="coerce")
    labels = labels.dropna(subset=["date", "symbol"]).copy()
    pooled = labels[
        (labels["freq"] == "YE")
        & (labels["k"].isin([int(k) for k in list(config.get("rf_ye_list") or [])]))
    ].copy()
    if pooled.empty:
        empty = pd.DataFrame()
        return empty, empty, empty
    by_symbol = (
        pooled.groupby("symbol")
        .agg(
            label_rows=("symbol", "size"),
            unique_dates=("date", "nunique"),
            long_labels=("label", lambda s: int((s == 1).sum())),
            short_labels=("label", lambda s: int((s == 0).sum())),
            first_label_date=("date", "min"),
            last_label_date=("date", "max"),
        )
        .reset_index()
        .sort_values(["label_rows", "unique_dates", "symbol"], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    by_symbol["first_label_date"] = by_symbol["first_label_date"].dt.strftime("%Y-%m-%d")
    by_symbol["last_label_date"] = by_symbol["last_label_date"].dt.strftime("%Y-%m-%d")
    by_k = (
        pooled.groupby("k")
        .agg(
            label_rows=("k", "size"),
            unique_symbols=("symbol", "nunique"),
            unique_dates=("date", "nunique"),
            long_labels=("label", lambda s: int((s == 1).sum())),
            short_labels=("label", lambda s: int((s == 0).sum())),
        )
        .reset_index()
        .sort_values("k")
        .reset_index(drop=True)
    )
    by_year = (
        pooled.assign(year=pooled["date"].dt.year)
        .groupby("year")
        .agg(
            label_rows=("year", "size"),
            unique_symbols=("symbol", "nunique"),
            long_labels=("label", lambda s: int((s == 1).sum())),
            short_labels=("label", lambda s: int((s == 0).sum())),
        )
        .reset_index()
        .sort_values("year")
        .reset_index(drop=True)
    )
    return by_symbol, by_k, by_year


def build_rf_feature_coverage_tables(feature_coverage, feature_metadata, config):
    threshold = float(config.get("rf_min_feature_coverage_pct", 10.0))
    if feature_coverage is None or feature_coverage.empty:
        empty = pd.DataFrame()
        return empty, empty, empty
    coverage = feature_coverage.copy()
    coverage["feature"] = coverage["feature"].astype(str)
    coverage["coverage_pct"] = pd.to_numeric(coverage["coverage_pct"], errors="coerce")
    coverage = coverage.dropna(subset=["feature", "coverage_pct"]).sort_values("coverage_pct", ascending=False).reset_index(drop=True)
    coverage["kept_for_training"] = coverage["coverage_pct"] >= threshold
    overview = pd.DataFrame(
        [
            {
                "coverage_threshold_pct": threshold,
                "active_features_before_filter": int(len(coverage)),
                "kept_features": int(coverage["kept_for_training"].sum()),
                "dropped_features": int((~coverage["kept_for_training"]).sum()),
                "dense_features_ge_50pct": int((coverage["coverage_pct"] >= 50.0).sum()),
                "mid_features_10_to_50pct": int(((coverage["coverage_pct"] >= threshold) & (coverage["coverage_pct"] < 50.0)).sum()),
                "sparse_features_below_threshold": int((coverage["coverage_pct"] < threshold).sum()),
            }
        ]
    )
    family_rows = []
    assigned_features = set()
    for family, family_columns in sorted((feature_metadata or {}).get("feature_family_columns", {}).items()):
        family_set = {str(column) for column in list(family_columns or [])}
        family_df = coverage[coverage["feature"].isin(family_set)].copy()
        if family_df.empty:
            continue
        assigned_features.update(family_df["feature"].tolist())
        family_rows.append(
            {
                "family": family,
                "active_cols": int(len(family_df)),
                "kept_cols": int(family_df["kept_for_training"].sum()),
                "dropped_cols": int((~family_df["kept_for_training"]).sum()),
                "avg_coverage_pct": round(float(family_df["coverage_pct"].mean()), 2),
                "median_coverage_pct": round(float(family_df["coverage_pct"].median()), 2),
            }
        )
    unassigned = coverage[~coverage["feature"].isin(assigned_features)].copy()
    if not unassigned.empty:
        family_rows.append(
            {
                "family": "unassigned",
                "active_cols": int(len(unassigned)),
                "kept_cols": int(unassigned["kept_for_training"].sum()),
                "dropped_cols": int((~unassigned["kept_for_training"]).sum()),
                "avg_coverage_pct": round(float(unassigned["coverage_pct"].mean()), 2),
                "median_coverage_pct": round(float(unassigned["coverage_pct"].median()), 2),
            }
        )
    family_table = pd.DataFrame(family_rows).sort_values(["kept_cols", "avg_coverage_pct"], ascending=[False, False]).reset_index(drop=True)
    lowest_coverage = coverage.sort_values("coverage_pct", ascending=True).head(20).reset_index(drop=True)
    return overview, family_table, lowest_coverage


class DirectSignalStrategy:
    def __init__(
        self,
        *,
        name,
        signal_col,
        price_col="close",
        dollar_volume_col="dollar_volume",
        rebalance_freq="M",
        gross_exposure=1.0,
        selection_side="long_short",
        action_transform="sign",
        action_threshold=0.0,
        vol_scale_col=None,
        inverse_vol_scale=False,
    ):
        self._name = str(name)
        self.signal_col = str(signal_col)
        self.price_col = str(price_col)
        self.dollar_volume_col = str(dollar_volume_col)
        self.rebalance_freq = str(rebalance_freq).upper()
        self.gross_exposure = float(gross_exposure)
        self.selection_side = str(selection_side).lower()
        self.action_transform = str(action_transform).lower()
        self.action_threshold = float(action_threshold)
        self.vol_scale_col = None if vol_scale_col in (None, "") else str(vol_scale_col)
        self.inverse_vol_scale = bool(inverse_vol_scale)

    @property
    def name(self):
        return self._name

    def _rebalance_dates(self, date_index):
        if self.rebalance_freq == "D":
            return set(date_index)
        if self.rebalance_freq == "M":
            return set(pd.Series(date_index, index=date_index).groupby(date_index.to_period("M")).head(1).tolist())
        return set(pd.Series(date_index, index=date_index).groupby(date_index.to_period("W")).head(1).tolist())

    def _normalized_weights(self, signal_row, eligible_row, risk_row=None):
        base = pd.to_numeric(signal_row, errors="coerce").where(eligible_row, 0.0).fillna(0.0)
        if self.action_threshold > 0.0:
            base = base.where(base.abs() >= self.action_threshold, 0.0)
        if self.action_transform == "sign":
            base = base.gt(0.0).astype(float) - base.lt(0.0).astype(float)
        elif self.action_transform == "long_only":
            base = base.clip(lower=0.0)
        if self.selection_side == "long_only":
            base = base.clip(lower=0.0)
        else:
            base = base.clip(lower=-1.0, upper=1.0)
        if self.inverse_vol_scale and risk_row is not None:
            vol = pd.to_numeric(risk_row, errors="coerce")
            inv_vol = 1.0 / vol.where(vol > 1e-12)
            inv_vol = inv_vol.replace([np.inf, -np.inf], np.nan)
            base = (base * inv_vol).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        gross = float(base.abs().sum())
        if gross <= 0.0:
            return pd.Series(0.0, index=base.index, dtype=float)
        return (base * (self.gross_exposure / gross)).astype(float)

    def compute_weights(self, panel):
        dates = pd.DatetimeIndex(sorted(pd.Index(panel.index.get_level_values("date")).unique()))
        symbols = sorted(pd.Index(panel.index.get_level_values("symbol")).unique())
        weights = pd.DataFrame(0.0, index=dates, columns=symbols)
        if len(dates) == 0 or len(symbols) == 0:
            return weights
        signal_panel = panel[self.signal_col].unstack("symbol").reindex(index=dates, columns=symbols)
        price_panel = panel[self.price_col].unstack("symbol").reindex(index=dates, columns=symbols)
        vol_panel = None
        if self.vol_scale_col and self.vol_scale_col in panel.columns:
            vol_panel = panel[self.vol_scale_col].unstack("symbol").reindex(index=dates, columns=symbols)
        rebalance_dates = self._rebalance_dates(dates)
        current = pd.Series(0.0, index=symbols, dtype=float)
        for dt in dates:
            if dt in rebalance_dates:
                eligible = signal_panel.loc[dt].notna()
                risk_row = vol_panel.loc[dt] if vol_panel is not None else None
                current = self._normalized_weights(signal_panel.loc[dt], eligible, risk_row=risk_row)
            weights.loc[dt] = current.reindex(symbols).fillna(0.0)
        return weights


def summarize_series(returns, turnover, trade_count):
    returns = pd.Series(returns, dtype=float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if returns.empty:
        return {
            "days": 0,
            "start_date": "",
            "end_date": "",
            "sharpe": 0.0,
            "total_return": 0.0,
            "final_equity": 1.0,
            "max_drawdown": 0.0,
            "avg_turnover": 0.0,
            "total_turnover": 0.0,
            "trade_count": int(trade_count),
            "positive_days": 0,
            "negative_days": 0,
        }
    equity = (1.0 + returns).cumprod()
    peak = equity.cummax()
    drawdown = (equity / peak) - 1.0
    volatility = float(returns.std(ddof=0)) if len(returns) > 1 else 0.0
    sharpe = (float(returns.mean()) / volatility * math.sqrt(252.0)) if volatility > 1e-12 else 0.0
    return {
        "days": int(len(returns)),
        "start_date": str(pd.Timestamp(returns.index.min()).date()),
        "end_date": str(pd.Timestamp(returns.index.max()).date()),
        "sharpe": float(sharpe),
        "total_return": float(equity.iloc[-1] - 1.0),
        "final_equity": float(equity.iloc[-1]),
        "max_drawdown": float(drawdown.min()),
        "avg_turnover": float(pd.Series(turnover, dtype=float).mean()) if len(turnover) else 0.0,
        "total_turnover": float(pd.Series(turnover, dtype=float).sum()) if len(turnover) else 0.0,
        "trade_count": int(trade_count),
        "positive_days": int((returns > 0.0).sum()),
        "negative_days": int((returns < 0.0).sum()),
    }


def count_symbol_weight_changes(weights):
    if weights.empty:
        return 0
    delta = weights.fillna(0.0).diff().fillna(weights.fillna(0.0)).abs()
    return int((delta > 1e-12).sum().sum())


def build_paper_tsmom_panel(feature_frame, config):
    signal_col = first_available_column(feature_frame.columns, config["tsmom_signal_col_candidates"])
    vol_col = first_available_column(feature_frame.columns, config["tsmom_vol_col_candidates"])
    if not signal_col:
        raise RuntimeError("Could not find a 12-month return column for the paper-style TSMOM signal.")
    panel = feature_frame[[column for column in ["date", "symbol", "close", "dollar_volume", signal_col, vol_col] if column in feature_frame.columns]].copy()
    panel["tsmom_signal"] = pd.to_numeric(panel[signal_col], errors="coerce")
    panel["tsmom_vol"] = pd.to_numeric(panel[vol_col], errors="coerce") if vol_col else np.nan
    panel["close"] = pd.to_numeric(panel["close"], errors="coerce")
    panel["dollar_volume"] = pd.to_numeric(panel.get("dollar_volume"), errors="coerce")
    panel = panel.dropna(subset=["date", "symbol", "close", "tsmom_signal"]).copy()
    panel = panel.set_index(["date", "symbol"]).sort_index()
    diagnostics = {
        "signal_column_used": signal_col,
        "volatility_column_used": vol_col,
        "rows": int(len(panel)),
        "symbols": int(panel.index.get_level_values("symbol").nunique()) if not panel.empty else 0,
    }
    return panel, diagnostics


def build_rf_panel(feature_frame, rf_scored_frame, config):
    vol_col = first_available_column(feature_frame.columns, config["tsmom_vol_col_candidates"])
    vol_frame = feature_frame[[column for column in ["date", "symbol", vol_col] if column in feature_frame.columns]].copy()
    if vol_col:
        vol_frame = vol_frame.rename(columns={vol_col: "rf_vol"})
    panel = rf_scored_frame.copy()
    panel = panel.merge(vol_frame, on=["date", "symbol"], how="left")
    panel["close"] = pd.to_numeric(panel["close"], errors="coerce")
    panel["dollar_volume"] = pd.to_numeric(panel.get("dollar_volume"), errors="coerce")
    panel["rf_signal"] = pd.to_numeric(panel.get("rf_signal"), errors="coerce")
    panel["rf_vol"] = pd.to_numeric(panel.get("rf_vol"), errors="coerce")
    panel = panel.dropna(subset=["date", "symbol", "close", "rf_signal"]).copy()
    panel = panel.set_index(["date", "symbol"]).sort_index()
    diagnostics = {
        "rows": int(len(panel)),
        "symbols": int(panel.index.get_level_values("symbol").nunique()) if not panel.empty else 0,
        "avg_signal_contributors": float(pd.to_numeric(rf_scored_frame.get("rf_signal_contributors"), errors="coerce").mean()),
    }
    return panel, diagnostics


def normalize_equity_frame(df):
    out = pd.DataFrame(index=df.index)
    for column in df.columns:
        series = pd.to_numeric(df[column], errors="coerce")
        valid = series.dropna()
        if valid.empty:
            out[column] = np.nan
            continue
        first = float(valid.iloc[0])
        out[column] = series / first if abs(first) > 1e-12 else np.nan
    return out.ffill()


In [ ]:
rf_training_summary, rf_model_availability = ensure_rf_models(NOTEBOOK_CONFIG)
print("RF model availability")
display(rf_model_availability)
if isinstance(rf_training_summary, dict):
    print("RF training status:", rf_training_summary.get("status", "trained"))
rf_training_payload, rf_feature_coverage, rf_label_frame = load_rf_training_artifacts(NOTEBOOK_CONFIG, rf_training_summary)
rf_training_objective = build_rf_training_objective_table(NOTEBOOK_CONFIG, rf_training_payload)
rf_label_diagnostics = build_rf_label_diagnostics_table(rf_label_frame, rf_training_payload)
rf_label_by_symbol, rf_label_by_k, rf_label_by_year = build_rf_label_breakdown_tables(rf_label_frame, NOTEBOOK_CONFIG)
rf_coverage_overview, rf_coverage_by_family, rf_low_coverage_features = build_rf_feature_coverage_tables(
    rf_feature_coverage,
    (rf_training_payload or {}).get("feature_metadata", {}),
    NOTEBOOK_CONFIG,
)
print("RF training objective")
display(rf_training_objective)


In [ ]:
print("RF label diagnostics")
display(rf_label_diagnostics)
print("RF labels by k")
display(rf_label_by_k)
print("RF labels by symbol")
display(rf_label_by_symbol)
print("RF labels by year")
display(rf_label_by_year)


In [ ]:
print("RF feature coverage overview")
display(rf_coverage_overview)
print("RF feature coverage by family")
display(rf_coverage_by_family)
print("Lowest-coverage RF features")
display(rf_low_coverage_features)


In [ ]:
feature_frame, price_diagnostics, feature_metadata = build_optimal_trade_feature_frame(
    symbols=NOTEBOOK_CONFIG["symbols"],
    start_date=NOTEBOOK_CONFIG["train_start_date"],
    end_date=NOTEBOOK_CONFIG["backtest_end_date"],
    download_missing_prices=bool(NOTEBOOK_CONFIG.get("fetch_missing_from_fmp", True)),
)
if feature_frame.empty:
    raise RuntimeError("No feature rows were built for the requested ETF universe.")

ts_panel, tsmom_diagnostics = build_paper_tsmom_panel(feature_frame, NOTEBOOK_CONFIG)
rf_scored_frame, rf_score_diagnostics = score_optimal_trade_random_forest_models(
    feature_frame,
    model_name_prefix=NOTEBOOK_CONFIG["rf_model_name_prefix"],
    ye_list=NOTEBOOK_CONFIG["rf_ye_list"],
    complete_case=False,
)
rf_panel, rf_panel_diagnostics = build_rf_panel(feature_frame, rf_scored_frame, NOTEBOOK_CONFIG)
comparison_panel = ts_panel.join(
    rf_panel[[column for column in ["rf_signal", "rf_signal_contributors", "rf_vol"] if column in rf_panel.columns]],
    how="left",
)

print("Price diagnostics")
display(price_diagnostics)
print("Paper-style TSMOM diagnostics")
display(pd.DataFrame([tsmom_diagnostics]))
print("RF score diagnostics")
display(rf_score_diagnostics)
print("RF panel diagnostics")
display(pd.DataFrame([rf_panel_diagnostics]))


In [ ]:
period_start = pd.Timestamp(NOTEBOOK_CONFIG["backtest_start_date"])
period_end = pd.Timestamp(NOTEBOOK_CONFIG["backtest_end_date"])
period_mask = (
    (comparison_panel.index.get_level_values("date") >= period_start)
    & (comparison_panel.index.get_level_values("date") <= period_end)
)
period_panel = comparison_panel.loc[period_mask].copy()
if period_panel.empty:
    raise RuntimeError(f"No rows were available for evaluation window {period_start.date()} to {period_end.date()}.")

tsmom_eval_panel = period_panel.dropna(subset=["tsmom_signal"]).copy()
rf_eval_panel = period_panel.dropna(subset=["rf_signal"]).copy()
benchmark_panel = period_panel.dropna(subset=["close"]).copy()

tsmom_strategy = DirectSignalStrategy(
    name="PaperStyleTSMOM12M",
    signal_col="tsmom_signal",
    action_transform="sign",
    vol_scale_col="tsmom_vol",
    inverse_vol_scale=True,
    rebalance_freq=NOTEBOOK_CONFIG["rebalance_freq"],
    gross_exposure=NOTEBOOK_CONFIG["gross_exposure"],
)
rf_strategy = DirectSignalStrategy(
    name="OptimalTradeRFClassifier",
    signal_col="rf_signal",
    action_transform="identity",
    vol_scale_col="rf_vol",
    inverse_vol_scale=True,
    rebalance_freq=NOTEBOOK_CONFIG["rebalance_freq"],
    gross_exposure=NOTEBOOK_CONFIG["gross_exposure"],
)
benchmark_strategy = BuyAndHoldEqualWeightStrategy(
    price_col="close",
    gross_exposure=float(NOTEBOOK_CONFIG["gross_exposure"]),
    top_k=None,
    liquidate_on_last_day=True,
)
execution_cfg = ExecutionConfig(
    price_col="close",
    fee_bps=float(NOTEBOOK_CONFIG["fee_bps"]),
    slippage_bps=float(NOTEBOOK_CONFIG["slippage_bps"]),
    use_lagged_weights=True,
    turnover_half_l1=True,
)

tsmom_weights = tsmom_strategy.compute_weights(tsmom_eval_panel)
rf_weights = rf_strategy.compute_weights(rf_eval_panel)

tsmom_result = backtest_panel(panel=tsmom_eval_panel, strategy=tsmom_strategy, cfg=execution_cfg)
rf_result = backtest_panel(panel=rf_eval_panel, strategy=rf_strategy, cfg=execution_cfg)
benchmark_result = backtest_panel(panel=benchmark_panel, strategy=benchmark_strategy, cfg=execution_cfg)

strategy_runs = [
    ("Paper TSMOM", tsmom_strategy, tsmom_result, tsmom_eval_panel, tsmom_weights),
    ("Optimal-Trade RF", rf_strategy, rf_result, rf_eval_panel, rf_weights),
    ("Equal-Weight Benchmark", benchmark_strategy, benchmark_result, benchmark_panel, benchmark_strategy.compute_weights(benchmark_panel)),
]
summary_rows = []
for label, strategy_obj, result_obj, panel_used, weights_used in strategy_runs:
    metrics = summarize_series(result_obj.returns, result_obj.turnover, count_symbol_weight_changes(weights_used))
    summary_rows.append(
        {
            "strategy": label,
            "strategy_name": strategy_obj.name,
            "backtest_start_date": str(period_start.date()),
            "backtest_end_date": str(period_end.date()),
            "rows_used": int(len(panel_used)),
            "symbols_used": int(panel_used.index.get_level_values("symbol").nunique()) if not panel_used.empty else 0,
            "trade_count": int(metrics["trade_count"]),
            "cumulative_return": float(metrics["total_return"]),
            "sharpe": float(metrics["sharpe"]),
            "max_drawdown": float(metrics["max_drawdown"]),
            "avg_turnover": float(metrics["avg_turnover"]),
            "total_turnover": float(metrics["total_turnover"]),
            "positive_days": int(metrics["positive_days"]),
            "negative_days": int(metrics["negative_days"]),
        }
    )
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

equity_df = pd.DataFrame(
    {
        "Paper TSMOM": tsmom_result.equity_curve,
        "Optimal-Trade RF": rf_result.equity_curve,
        "Equal-Weight Benchmark": benchmark_result.equity_curve,
    }
).dropna(how="all")
if not equity_df.empty:
    normalized = normalize_equity_frame(equity_df)
    ax = normalized.plot(figsize=(12, 5), linewidth=2, color=["darkgreen", "darkorange", "steelblue"])
    ax.set_title(f"Paper-Style TSMOM vs Optimal-Trade RF ({period_start.date()} to {period_end.date()})")
    ax.set_xlabel("Date")
    ax.set_ylabel("Growth of $1")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

report_lines = [
    "## Notebook Summary",
    "",
    f"- Backtest window: `{period_start.date()}` to `{period_end.date()}`.",
    "- Paper-style TSMOM implementation used here: sign of the trailing 12-month return with monthly rebalancing and inverse-volatility scaling.",
    "- This intentionally does **not** use the earlier `((1 + 12m) / (1 + 1m)) - 1` proxy.",
    "- RF strategy used a pre-2020 oracle-trained pooled classifier built from `YE = [1, 2, 4, 8]`, with the signal defined as `2 * P(long) - 1`.",
    "",
    summary_df.to_markdown(index=False),
]
display(Markdown("\n".join(report_lines)))
